# Day 6: Session 6C - A Date Is Not a Number

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6c_dates.html)

Date: 09/08/2026

In [1]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/toolik_weather.csv'
toolik = pd.read_csv(url)

toolik[['Year', 'Month', 'Date', 'Daily_AirTemp_Mean_C']].head()

,Year,Month,Date,Daily_AirTemp_Mean_C
0,1988,6,19880601,8.4
1,1988,6,19880602,6.0
2,1988,6,19880603,5.8
3,1988,6,19880604,1.8
4,1988,6,19880605,6.8


In [2]:
toolik['Date'].dtype

dtype('int64')

### the parsing pattern

pd.to_datetime() takes a column of dates in disguise and returns a column of real dates. You describe the disguise with format=:

In [4]:
toolik['date'] = pd.to_datetime(toolik['Date'], format='%Y%m%d')

toolik[['Date', 'date']].head()



# pd.to_datetime(column, format='%Y%m%d')
#                  ↑    |      ↑         
#       | what to parse | how it is laid out |

,Date,date
0,19880601,1988-06-01
1,19880602,1988-06-02
2,19880603,1988-06-03
3,19880604,1988-06-04
4,19880605,1988-06-05


In [5]:
print(toolik['date'].dtype)
print(toolik['date'].min())
print(toolik['date'].max())

datetime64[ns]
1988-06-01 00:00:00
2018-12-31 00:00:00


### Test your knowledge
The date_data.csv file at https://eds-217-essential-python.github.io/data/date_data.csv has a Date column stored as text in the form 2023-11-02. Read it in, write the parsing pattern for it with the correct format string, and print the dtype of the result to prove it worked.


In [10]:
from pandas import read_csv
url = 'https://eds-217-essential-python.github.io/data/date_data.csv'
df = read_csv(url)
df.head()

,Date,Value,Category
0,2023-11-02,19.51,A
1,2022-05-25,41.51,B
2,2021-10-05,70.52,C
3,2023-07-16,7.61,A
4,2021-01-08,30.93,C


In [17]:
df['date'] = pd.to_datetime(toolik['Date'], format='%Y%m%d')
df[['Date', 'date']].head()

,Date,date
0,2023-11-02,1988-06-01
1,2022-05-25,1988-06-02
2,2021-10-05,1988-06-03
3,2023-07-16,1988-06-04
4,2021-01-08,1988-06-05


In [15]:
print(df['date'].dtype)

datetime64[ns]


### The .dt accessors



In [18]:
toolik['year'] = toolik['date'].dt.year
toolik['month'] = toolik['date'].dt.month
toolik['day'] = toolik['date'].dt.day

toolik[['date', 'year', 'month', 'day']].head()

,date,year,month,day
0,1988-06-01,1988,6,1
1,1988-06-02,1988,6,2
2,1988-06-03,1988,6,3
3,1988-06-04,1988,6,4
4,1988-06-05,1988,6,5


In [19]:
# proving that .dt worked
print((toolik['year'] == toolik['Year']).all())
print((toolik['month'] == toolik['Month']).all())

True
True


### test your knowledge
Add a dayofyear column to toolik using the .dt accessor for it. 


In [21]:
toolik['dayofyear'] = toolik['date'].dt.dayofyear
toolik.head()

,Year,Month,Date,LTER_Site,Station,Daily_AirTemp_Mean_C,Flag_Daily_AirTemp_Mean_C,Daily_AirTemp_AbsMax_C,Flag_Daily_AirTemp_AbsMax_C,Daily_AirTemp_AbsMin_C,...,Daily_globalrad_total_jcm2,FLAG_Daily_globalrad_total_mjm2,Moss,Soil20cm,Comments,date,year,month,day,dayofyear
0,1988,6,19880601,ARC,TLKMAIN,8.4,E,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Air temp 1 & 5 meter estimated from regressio...,1988-06-01,1988,6,1,153
1,1988,6,19880602,ARC,TLKMAIN,6.0,E,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Air temp 1 & 5 meter estimated from regressio...,1988-06-02,1988,6,2,154
2,1988,6,19880603,ARC,TLKMAIN,5.8,E,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Air temp 1 & 5 meter estimated from regressio...,1988-06-03,1988,6,3,155
3,1988,6,19880604,ARC,TLKMAIN,1.8,E,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Air temp 1 & 5 meter estimated from regressio...,1988-06-04,1988,6,4,156
4,1988,6,19880605,ARC,TLKMAIN,6.8,E,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,Air temp 1 & 5 meter estimated from regressio...,1988-06-05,1988,6,5,157


### Dates as grouping keys



In [22]:
toolik.groupby('month')['Daily_AirTemp_Mean_C'].mean().round(2)

month
1    -22.89
2    -20.70
3    -20.69
4    -11.76
5     -0.80
6      8.59
7     11.22
8      7.23
9     -0.11
10   -10.54
11   -18.34
12   -21.44
Name: Daily_AirTemp_Mean_C, dtype: float64

In [23]:
toolik.groupby('year')['Daily_AirTemp_Mean_C'].agg(['count', 'mean']).head(3).round(2)

,count,mean
year,,
1988,214,-4.88
1989,365,-7.94
1990,365,-8.54


### The question that needs a month column

Toolik has thirty years of data, and the obvious question to ask is whether the weather is getting warmer.

Split the record into its first eleven years and its last ten years, using Wednesday’s filter pattern (we are leaving 1999 to 2008 out of both halves, so the two ends of the record are well separated):



In [24]:
early = toolik[toolik['year'] <= 1998]
late = toolik[toolik['year'] >= 2009]

print(early.shape)
print(late.shape)

(3866, 26)
(3652, 26)


In [26]:
# after splitting the record into "early" and "late",
# ask each half the same question, and put the two answers side by side:

comparison = pd.DataFrame({
    'early': early.groupby('month')['Daily_AirTemp_Mean_C'].mean(),
    'late': late.groupby('month')['Daily_AirTemp_Mean_C'].mean(),
})

comparison['change'] = comparison['late'] - comparison['early']

comparison.round(2)

,early,late,change
month,,,
1,-24.72,-21.26,3.46
2,-21.86,-19.38,2.48
3,-18.62,-20.64,-2.02
4,-10.40,-11.86,-1.45
5,0.37,-0.54,-0.91
6,8.43,8.19,-0.24
7,12.05,11.28,-0.77
8,7.48,7.18,-0.30
9,-0.85,-0.01,0.84
